In [1]:
'''
!/usr/bin/env python
-*- encoding: utf-8 -*-
---------------------------------------------------
Author: Yuqi Shi syq010316@gmail.com
Date: 2023-04-06 05:10:19
LastEditors: Yuqi Shi syq010316@gmail.com
LastEditTime: 2023-04-06 05:10:28
FilePath: /metabolism/model_evaluation.ipynb
Description: a notebook to evaluate the model
---------------------------------------------------
Copyright (c) 2023 by Yuqi Shi, All Rights Reserved. 
'''


'\n!/usr/bin/env python\n-*- encoding: utf-8 -*-\n---------------------------------------------------\nAuthor: Yuqi Shi syq010316@gmail.com\nDate: 2023-04-06 05:10:19\nLastEditors: Yuqi Shi syq010316@gmail.com\nLastEditTime: 2023-04-06 05:10:28\nFilePath: /metabolism/model_evaluation.ipynb\nDescription: a notebook to evaluate the model\n---------------------------------------------------\nCopyright (c) 2023 by Yuqi Shi, All Rights Reserved. \n'

set model checkpoint and device

In [2]:
import torch

model_checkpoint = "/home/user-home/shiyuqi/metabolism/models/reaction_prediction_augmented/flan-t5-base-reaction-30"
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

load the datasets

In [3]:
from datasets import load_dataset

data_files = {'test': './datasets/test_reactions_dataset.csv'}
datasets = load_dataset('csv', data_files=data_files)

Found cached dataset csv (/home/user-home/shiyuqi/.cache/huggingface/datasets/csv/default-3513fe111c1e5647/0.0.0/6b34fb8fcf56f7c8ba51dc895bfa2bfbe43546f190a60fcf74bb5e8afdcc2317)


  0%|          | 0/1 [00:00<?, ?it/s]

tokenize the datasets

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('/home/user-home/shiyuqi/metabolism/models/flan-t5-base-tokenizer-vocab-5000', return_tensors="pt")

max_length = 1024
def preprocess_function(dataset):
    model_inputs = tokenizer(dataset['substrate'], max_length=max_length, truncation=True, padding=True)
    labels = tokenizer(dataset['metabolite'], max_length=max_length, truncation=True, padding=True)
    model_inputs['labels'] = labels['input_ids'] 
    return model_inputs

tokenized_datasets = datasets.map(preprocess_function, batched=True, remove_columns=datasets["test"].column_names)

Loading cached processed dataset at /home/user-home/shiyuqi/.cache/huggingface/datasets/csv/default-3513fe111c1e5647/0.0.0/6b34fb8fcf56f7c8ba51dc895bfa2bfbe43546f190a60fcf74bb5e8afdcc2317/cache-a41cc4ff88b1b837.arrow


load the model from the checkpoint

In [5]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint).to(device)

2023-05-03 06:35:10.808898: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-05-03 06:35:10.976525: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2023-05-03 06:35:11.551766: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: :/home/user-home/shiyuqi/software/usr/lib/x86_64-linux-gnu
2023-05-03 06:35:11.551843: 

customize dataloaders

In [6]:
from torch.utils.data import DataLoader
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

tokenized_datasets.set_format(type='torch')
test_dataloader = DataLoader(
    tokenized_datasets["test"],
    shuffle=False,
    collate_fn=data_collator,
    batch_size=1
)


set accelerator

In [7]:
from accelerate import Accelerator

accelerator = Accelerator()
model, test_dataloader = accelerator.prepare(model, test_dataloader)


define postprocess function

In [8]:
def postprocess(predictions):
    predictions = predictions.cpu().numpy()
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_preds = [pred.strip() for pred in decoded_preds]
    return decoded_preds

initialize the method comparison file

In [9]:
# with open("method_comparison.csv", "w") as f:
#     f.write("method,at_least_one_metabolite,at_least_half_metabolites,all_metabolites,precision,recall,f1_score,invalid,time\n")

evaluation (with beam search)

In [10]:
from rdkit import Chem
from tqdm.auto import tqdm
import pandas as pd
import time
import os


model.eval()
all_counts = 0
corr = 0 
at_least_one, at_least_half, all_metabolites = 0, 0, 0
len_preds, len_corr_preds, len_actual_metabolites, invaild = 0, 0, 0, 0

method = 'test1'
result_filename = os.path.join('./test_results', method + '.csv')
with open(result_filename, 'w') as f:
    f.write("substrate,metabolite,all_predictions,correct_predictions\n")

df = pd.read_csv(data_files['test'], header=0, encoding='utf-8')

i = 0
start_time = time.time()
for batch in tqdm(test_dataloader):
    with torch.no_grad():
        batch = {k: v.to(device) for k, v in batch.items()}
        generated_tokens = accelerator.unwrap_model(model).generate(
            batch["input_ids"],
            attention_mask=batch["attention_mask"],
            max_length=1024,
            num_beams=20,
            num_return_sequences=20,          
        )

    generated_tokens = accelerator.pad_across_processes(generated_tokens, dim=1, pad_index=tokenizer.pad_token_id)
    predictions_gathered = accelerator.gather(generated_tokens)

    decode_preds= postprocess(predictions_gathered)
    decode_preds = list(set(decode_preds))

    metabolites = df['metabolite'][i].strip().split('|')
    substrate = df['substrate'][i].strip()
    corr_preds = []
    for pred in decode_preds:
        try:
            pred = Chem.MolToSmiles(Chem.MolFromSmiles(pred))
        except:
            invaild += 1
            pass

        if pred in metabolites:
            corr_preds.append(pred)
            
    if len(corr_preds) == len(metabolites):
        all_metabolites += 1
    if len(corr_preds) >= len(metabolites) / 2:
        at_least_half += 1
    if len(corr_preds) >= 1:
        at_least_one += 1
   
    all_counts += 1

    with open(result_filename, 'a') as f:
        f.write(",".join([substrate, "|".join(metabolites), "|".join(decode_preds), "|".join(corr_preds)]) + "\n")

    len_actual_metabolites += len(metabolites)
    len_corr_preds += len(corr_preds)
    len_preds += len(decode_preds)
    i += 1

time = time.time() - start_time

with open("method_comparison.csv", 'a') as f:
    f.write(
        ",".join([method, 
                  "{:.2%}".format(at_least_one / all_counts), # at_least_one_metabolite
                  "{:.2%}".format(at_least_half / all_counts), # at_least_half_metabolites
                  "{:.2%}".format(all_metabolites / all_counts), # all_metabolites
                  "{:.2%}".format(len_corr_preds / len_preds), # precision
                  "{:.2%}".format(len_corr_preds / len_actual_metabolites), # recall
                  "{:.2f}".format(2 * len_corr_preds / (len_preds + len_actual_metabolites)), # f1_score
                  "{:.2%}".format(invaild / len_preds), # invalid
                  "{:.2f}s".format(time) # time
    ]) + "\n")






  0%|          | 0/84 [00:00<?, ?it/s]

You're using a T5TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
[06:35:25] SMILES Parse Error: extra open parentheses for input: 'C#C[C@]1(O[C@@H]2OC(C(=O)O)[C@@H](O)[C@H](O)[C@H]2O)CC[C@H]2[C@@H]3CCC4=C(C=C[N+](O)CC[C@@H]4[C@H]3CC[C@@]21CC'
[06:35:29] SMILES Parse Error: unclosed ring for input: 'C=CCC1(CC(C)C(=O)O[C@@H]2OC(C(=O)O)[C@@H](O)[C@H](O)[C@H]2O)C=C(C)C[C@@]2(C)C(=O)NC(=O)N(O)C1=O'
[06:35:29] SMILES Parse Error: unclosed ring for input: 'CC(CO)CCC1(CC(C)C(=O)NC(=O)NC1=O)C(=O)NC(=O)NC1=O'
[06:35:29] SMILES Parse Error: unclosed ring for input: 'C=CCC1(CC(C)C(=O)O[C@@H]2OC(C(=O)O)[C@@H](O)[C@H](O)[C@H]2O)C=C(C)C[C@@]1(CC(C)C)C(=O)NC(=O)NC1=O'
[06:35:29] SMILES Parse Error: unclosed ring for input: 'C=CCC1(CC(C)C(=O)O[C@@H]2OC(C(=O)O)[C@@H](O)[C@H](O)[C@H]2O)C=C(C)C2CC(O)CC1(CC(C)C)C(=O)NC(=O)NC1=O'
[06:35:29] SMILES Par